# 第07章 图像平滑处理（跟随代码）

> 《OpenCV轻松入门：面向Python》（李立宗）第 7 章跟随代码。**全量收录原书代码**（例 7.1~例 7.10 + 章内独立演示），缺失例题已按书补全，代码以网页笔记 ch07.md 为准（含笔误与参数修正）。
> `cv2.imshow` 需在**本地 Python 环境**运行（浏览器/Jupyter 内不弹窗）；运行前请确认**工作目录为素材根目录**（本文件夹，picture*.jpg 等素材都在这里），否则 `imread` 返回 None。
> 原 notebook 用到的 `cv_show` 是未给出定义的辅助函数，已在本 notebook 开头补上定义（本地运行显示窗口用）。

# 第七章 图像平滑处理

> 平滑处理 = 在尽量保留原图信息的前提下过滤噪声：把与周围差异大的噪声点调整为邻域像素的近似值（别名：模糊处理、图像滤波）。本章六种方法：均值滤波、方框滤波、高斯滤波（线性，压高斯噪声）、中值滤波（非线性，椒盐噪声克星）、双边滤波（保边缘）、2D 卷积（自定义核兜底）。

In [ ]:
# cv_show 辅助函数定义（本地运行显示窗口用）：原 notebook 直接使用了它但未给出定义
def cv_show(name, img):
    cv2.imshow(name, img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

import cv2  # 供 cv_show 与后续示例使用

## 7.1 均值滤波

用当前像素点周围 N×N 邻域内像素值的**均值**代替当前像素值；卷积核为全 1 矩阵乘 1/(M×N)。核越大去噪越强、失真越重。

`dst = cv2.blur(src, ksize[, anchor[, borderType]])`
- `src`：输入图像，通道数任意，各通道独立处理
- `ksize`：滤波核大小，如 `(5,5)` 表示 5×5 邻域均值（blur/boxFilter 对核尺寸**无奇数限制**，2×2 也合法）
- `anchor`：锚点，默认 `(-1,-1)`，表示锚点位于核中心
- `borderType`：边界填充方式，默认 `BORDER_DEFAULT`（= `BORDER_REFLECT_101`）；注意 blur 不支持 `BORDER_WRAP`

### 例 7.1 · 均值滤波去噪

读取噪声图像做 5×5 均值滤波，并排显示原始（噪声）图像与去噪结果。

In [ ]:
# ===== 7.1 例 7.1：均值滤波去噪 =====
import cv2

o = cv2.imread("picture5_gaussian.jpg")   # 读取待处理图像（原书 lenaNoise.png）
r = cv2.blur(o, (5, 5))                   # 5×5 邻域均值滤波
cv2.imshow("original", o)                 # 显示原始（噪声）图像
cv2.imshow("result", r)                   # 显示去噪图像
cv2.waitKey()
cv2.destroyAllWindows()

### 例 7.2 · 卷积核大小对比

用 5×5 与 30×30 两种核对同一幅噪声图像做均值滤波：**卷积核越大，参与均值运算的像素越多，去噪效果越好，计算时间越长，失真也越严重**，实际处理要在失真与去噪效果之间取得平衡。

In [ ]:
# ===== 7.1 例 7.2：卷积核大小对比 =====
import cv2

o = cv2.imread("picture5_gaussian.jpg")   # 原书 noise.bmp
r5 = cv2.blur(o, (5, 5))                  # 5×5 核均值滤波
r30 = cv2.blur(o, (30, 30))               # 30×30 核均值滤波
cv2.imshow("original", o)                 # 原始图像
cv2.imshow("result5", r5)                 # 5×5 滤波结果
cv2.imshow("result30", r30)               # 30×30 滤波结果
cv2.waitKey()
cv2.destroyAllWindows()

## 7.2 方框滤波

与均值滤波的不同在于**不计算像素均值**，可以自由选择是否归一化：`normalize=1` 时求邻域平均值（等效均值滤波）；`normalize=0` 时直接求邻域像素值之和，结果极易超过 255 被截断为纯白——它适合计算积分特征，而不是去噪。

`dst = cv2.boxFilter(src, ddepth, ksize[, anchor[, normalize[, borderType]]])`
- `src`：输入图像，通道数任意，各通道独立处理
- `ddepth`：输出图像的图像深度，一般用 `-1` 表示与原始图像相同
- `ksize`：滤波核大小，如 `(3,3)` 表示 3×3 邻域（**无奇数限制**）
- `anchor`：锚点，默认 `(-1,-1)`，表示锚点位于核中心
- `normalize`：是否归一化，默认 `True`（=1）
- `borderType`：边界填充方式，默认 `BORDER_DEFAULT`；注意 boxFilter 不支持 `BORDER_WRAP`

### 例 7.3 · 方框滤波（默认归一化）

省略 normalize 相当于 `normalize=1`，此时与 `cv2.blur(o,(5,5))` 完全等效。

In [ ]:
# ===== 7.2 例 7.3：方框滤波（默认归一化）=====
import cv2

o = cv2.imread("picture5_gaussian.jpg")   # 原书 lenaNoise.png
r = cv2.boxFilter(o, -1, (5, 5))          # normalize 取默认值 True，等效于 blur(o,(5,5))
cv2.imshow("original", o)                 # 原始图像
cv2.imshow("result", r)                   # 方框滤波结果
cv2.waitKey()
cv2.destroyAllWindows()

### 例 7.4 · 不归一化的方框滤波（5×5）

`normalize=0` 直接求 5×5 邻域像素值之和：基本都会超过 255 而被截断为 255，得到的图像接近纯白；少数点有颜色，是因为其周边邻域像素值都较小，相加后仍小于 255。

In [ ]:
# ===== 7.2 例 7.4：不归一化的方框滤波 =====
import cv2

o = cv2.imread("picture5_gaussian.jpg")           # 原书 lenaNoise.png
r = cv2.boxFilter(o, -1, (5, 5), normalize=0)     # 直接求 5×5 邻域像素值之和，不除以 25
cv2.imshow("original", o)                         # 原始图像
cv2.imshow("result", r)                           # 结果接近纯白
cv2.waitKey()
cv2.destroyAllWindows()

### 例 7.5 · 小核不归一化（2×2）

卷积核缩小到 2×2 后，四个像素值的和不一定大于 255，结果图像中只有部分像素点是白色的。

In [ ]:
# ===== 7.2 例 7.5：小核不归一化 =====
import cv2

o = cv2.imread("picture5_gaussian.jpg")           # 原书 lenaNoise.png
r = cv2.boxFilter(o, -1, (2, 2), normalize=0)     # 2×2 邻域求和：四个值的和不一定会超过 255
cv2.imshow("original", o)                         # 原始图像
cv2.imshow("result", r)                           # 部分像素不是白色
cv2.waitKey()
cv2.destroyAllWindows()

## 7.3 高斯滤波

邻域内每个像素**权重不再相等**：中心点权重大、离中心越远权重越小，在此基础上加权求和。核的宽高可以不同但**都必须是奇数**；未归一化的核实际使用时必须归一化，否则结果错误。

`dst = cv2.GaussianBlur(src, ksize, sigmaX[, sigmaY[, borderType]])`
- `src`：输入图像，通道数任意，各通道独立处理
- `ksize`：高斯核大小，**必须为奇数**（宽高可以不同），如 (3,5)、(5,5)、(7,7)
- `sigmaX`：X 方向（水平）标准差，控制权重比例；设为 0 时由 ksize 自动推算
- `sigmaY`：Y 方向（垂直）标准差，可选；sigmaY=0 时只采用 sigmaX，两者都为 0 时由 ksize 推算
- `borderType`：边界类型，默认值 `cv2.BORDER_DEFAULT`

### 例 7.6 · 高斯滤波去噪

5×5 核、`sigmaX=sigmaY=0`（由核尺寸自动推算）对噪声图像做高斯滤波。

In [ ]:
# ===== 7.3 例 7.6：高斯滤波去噪 =====
import cv2

o = cv2.imread("picture5_gaussian.jpg")     # 原书 lenaNoise.png
r = cv2.GaussianBlur(o, (5, 5), 0, 0)       # 5×5 核，sigmaX=sigmaY=0，由核尺寸自动推算
cv2.imshow("original", o)                   # 原始（噪声）图像
cv2.imshow("result", r)                     # 高斯滤波结果
cv2.waitKey()
cv2.destroyAllWindows()

## 7.4 中值滤波

**不采用加权求均值**：取邻域内所有像素值排序后的**中间值**（median）替代当前像素值。不做均值运算、不糊细节，是椒盐噪声的克星；代价是排序等操作带来较大的运算量。

`dst = cv2.medianBlur(src, ksize)`
- `src`：输入图像，通道数任意，各通道独立处理
- `ksize`：滤波核大小，**必须是大于 1 的奇数**（如 3、5、7）；注意这里是**单个整数**，不是 (3,3) 这样的元组

### 例 7.7 · 中值滤波去噪（先造 4% 椒盐噪声）

本机素材 picture1.jpg 本身是干净图像，"去噪"无的放矢，所以先撒 4% 椒盐噪声（`default_rng(7)` 固定随机种子保证结果可复现），再做 3×3 中值滤波。

In [ ]:
# ===== 7.4 例 7.7：中值滤波去噪 =====
import cv2
import numpy as np

o = cv2.imread("picture1.jpg")      # 原书 lenaNoise.png（本机素材本身无噪声）
rng = np.random.default_rng(7)      # 固定随机种子，结果可复现
rnd = rng.random(o.shape[:2])
n = o.copy()
n[rnd < 0.02] = 0                   # 2% 椒（黑点）
n[rnd > 0.98] = 255                 # 2% 盐（白点），合计 4% 椒盐噪声
r = cv2.medianBlur(n, 3)            # 3×3 邻域中值滤波
cv2.imshow("original", o)           # 原始图像
cv2.imshow("noisy", n)              # 加椒盐噪声后的图像
cv2.imshow("result", r)             # 中值滤波结果
cv2.waitKey()
cv2.destroyAllWindows()

## 7.5 双边滤波

综合考虑**空间距离**与**色彩差异**的滤波：距离越远权重越小、颜色差别越大权重越小。边缘两侧颜色差得远的点权重趋近 0 被忽略，因此能**保护边缘**；代价是去噪效果并不好。

`dst = cv2.bilateralFilter(src, d, sigmaColor, sigmaSpace[, borderType])`
- `src`：输入图像，通道数任意，各通道独立处理
- `d`：以当前像素点为中心的滤波直径；非正数时自动由 sigmaSpace 计算得到。d>5 时较慢：实时应用推荐 d=5，离线较大噪声可选 d=9
- `sigmaColor`：颜色差值范围，与当前点像素值差值小于 sigmaColor 的点才参与滤波；为 0 滤波失去意义，为 255 时直径内所有点都参与
- `sigmaSpace`：坐标空间中的 sigma 值，越大说明越多点能参与滤波计算
- `borderType`：边界样式，一般用默认值；除 borderType 外其余参数全部必选

### 例 7.8 · 双边滤波去噪

d=25、双 sigma=100 对噪声图像做双边滤波：噪声仍在，去噪效果并不好——它的优势体现在对边缘信息的保护上。

In [ ]:
# ===== 7.5 例 7.8：双边滤波去噪 =====
import cv2

o = cv2.imread("picture5_gaussian.jpg")     # 原书 lenaNoise.png
r = cv2.bilateralFilter(o, 25, 100, 100)    # 直径 25，颜色 sigma=100，空间 sigma=100
cv2.imshow("original", o)                   # 原始（噪声）图像
cv2.imshow("result", r)                     # 双边滤波结果
cv2.waitKey()
cv2.destroyAllWindows()

### 例 7.9 · 高斯 vs 双边，边缘保持对比

对同一幅黑白边缘测试图分别做 55×55 高斯滤波与 d=55 双边滤波：高斯滤波的边缘被模糊虚化，双边滤波的边缘得到较好保留。

> 原书代码写作链式赋值 `g=r=cv2.GaussianBlur(...)`，把同一个结果赋给 g 和 r 而 r 并未使用，功能不出错但属于多余写法，此处已按笔记修正。

In [ ]:
# ===== 7.5 例 7.9：高斯 vs 双边，边缘保持对比 =====
import cv2

o = cv2.imread("picture3.jpg")                    # 原书 bilTest.bmp（黑白边缘测试图）
g = cv2.GaussianBlur(o, (55, 55), 0, 0)           # 高斯滤波：55×55 大核
b = cv2.bilateralFilter(o, 55, 100, 100)          # 双边滤波：直径 55
cv2.imshow("original", o)                         # 原始图像
cv2.imshow("Gaussian", g)                         # 高斯滤波结果
cv2.imshow("bilateral", b)                        # 双边滤波结果
cv2.waitKey()
cv2.destroyAllWindows()

## 7.6 2D 卷积

前面的滤波函数都无法把卷积核确定为特定形式；允许用户**自定义卷积核**实现卷积操作的函数是 `cv2.filter2D()`。注意它实际做的是**相关**运算（核不翻转），核对称时（均值核、高斯核）与真卷积结果一致。

`dst = cv2.filter2D(src, ddepth, kernel[, anchor[, delta[, borderType]]])`
- `src`：输入图像，通道数任意，各通道独立处理
- `ddepth`：输出图像的图像深度，一般用 `-1` 表示与原始图像相同
- `kernel`：卷积核，一个单通道数组；想让彩色图每个通道用不同的核，必须先 `cv2.split` 拆分通道分别操作
- `anchor`：锚点，默认 `(-1,-1)`，即核中心
- `delta`：修正值，可选项；存在时在基础滤波结果上加上该值作为最终结果
- `borderType`：边界填充方式，通常用默认值

### 例 7.10 · 自定义卷积核

用 `np.ones((9,9),np.float32)/81` 构造 9×9 全 1 归一化核（权重和为 1），效果与 9×9 均值滤波 `cv2.blur(o,(9,9))` 一致；实际应用中可以定义更复杂的卷积核实现自定义滤波。

In [ ]:
# ===== 7.6 例 7.10：自定义卷积核 =====
import cv2
import numpy as np

o = cv2.imread("picture3.jpg")                # 原书 lena.bmp
kernel = np.ones((9, 9), np.float32) / 81     # 9×9 全 1 核，除以 81 归一化，权重和为 1
r = cv2.filter2D(o, -1, kernel)               # 用自定义核对图像做滤波
cv2.imshow("original", o)                     # 原始图像
cv2.imshow("result", r)                       # 滤波结果（9×9 邻域均值效果）
cv2.waitKey()
cv2.destroyAllWindows()

#### 独立演示 · 彩色图各通道异核滤波

彩色图想让每个通道用不同的核，必须先 `cv2.split` 拆分通道、分别 `filter2D`、再 `cv2.merge` 合回。下面给 B/G/R 三个通道分别用 9×9 均值核、伪锐化核、3×3 均值核滤波。

In [ ]:
# ===== 独立演示：彩色图各通道异核滤波（先 split 再分别 filter2D 后 merge）=====
import cv2
import numpy as np

kernel_1=np.ones((9,9),np.float32)/81     # 9×9 全 1 核：81 个元素，归一化系数应为 1/81（原 /25 未归一，且 uint8 装不下小数）
kernel_2=np.array([[0,1,0],[1,5,1],[0,1,0]])/9
kernel_3=np.ones((3,3),np.float32)/9

img=cv2.imread("picture3.jpg")
b,g,r=cv2.split(img)


dst1=cv2.filter2D(b,-1,kernel_1)
dst2=cv2.filter2D(g,-1,kernel_2)
dst3=cv2.filter2D(r,-1,kernel_3)

dst=cv2.merge((dst1,dst2,dst3))
cv_show("original",img)
cv_show("filtered",dst)

### 拓展验证 · filter2D 与手算卷积对照

`filter2D` 做的是相关（核不翻转）；对称核（均值核、高斯核）与真卷积结果一致。下面用 3×3 均值核对 5×5 已知数组滤波，与逐点手算的 3×3 邻域均值对照，内部区域完全一致（输出 True）。

In [ ]:
# ===== 7.6 拓展验证：filter2D 与手算卷积对照 =====
import cv2
import numpy as np

a = np.arange(25, dtype=np.float32).reshape(5, 5)      # 5×5 已知数组
k = np.ones((3, 3), np.float32) / 9.0                  # 3×3 均值核（对称核）
out = cv2.filter2D(a, -1, k, borderType=cv2.BORDER_ISOLATED)  # 不考虑边界外推
hand = np.zeros_like(a)
for i in range(1, 4):                                  # 只算内部 3×3 区域，避免边界干扰
    for j in range(1, 4):
        hand[i, j] = a[i-1:i+2, j-1:j+2].sum() / 9.0   # 手算：3×3 窗口均值
print(np.allclose(out[1:4, 1:4], hand[1:4, 1:4]))      # True：filter2D 与手算一致